# 🟤 Copper Spot Price Forecasting Model

**Professional-grade multi-model ensemble for copper price forecasting**

This notebook provides an end-to-end, production-quality pipeline:

| Section | Content |
|---|---|
| 1 | Setup & configuration |
| 2 | Data ingestion (yfinance + FRED + COT) |
| 3 | Exploratory data analysis |
| 4 | Cointegration analysis & regime detection |
| 5 | Feature engineering pipeline |
| 6 | Model training & Optuna tuning |
| 7 | Feature pruning (SHAP-based) |
| 8 | Walk-forward cross-validation |
| 9 | Out-of-sample backtesting |
| 10 | Forecast with 80% confidence interval |
| 11 | SHAP explainability |
| 12 | Interactive Plotly visualisations |
| 13 | Scenario analysis |
| 14 | Export results |

---
> **Data sources**: [yfinance](https://github.com/ranaroussi/yfinance) (free) · [FRED API](https://fred.stlouisfed.org/docs/api/fred/) (free) · [CFTC COT](https://www.cftc.gov/MarketReports/CommitmentsofTraders/index.htm) (via Nasdaq Data Link)
>
> **Models**: Naive (RW) · Ridge · XGBoost · LightGBM · ARIMAX · Prophet · Hybrid (ARIMAX+LGBM) · Weighted Ensemble · Stacking Ensemble

## 1. Setup & Configuration


In [ ]:
import os
import sys
import warnings
import logging
import numpy as np
import pandas as pd

warnings.filterwarnings('ignore')
logging.basicConfig(level=logging.INFO, format='%(levelname)s | %(message)s')

# Add project root to path
sys.path.insert(0, os.path.abspath('..'))

from src.data_ingestion import load_data
from src.feature_engineering import build_features, split_features_targets
from src.models import NaiveModel, LinearModel, XGBoostModel, LGBMModel, EnsembleModel, QuantileForecaster
from src.models_arimax import ARIMAXModel
from src.models_hybrid import HybridModel
from src.models_stacking import StackingEnsemble
from src.evaluation import compute_metrics, walk_forward_cv, compare_models, out_of_sample_backtest
from src.visualization import (
    plot_price_history, plot_feature_correlations, plot_cv_results,
    plot_forecast_with_ci, plot_model_comparison, plot_shap_summary,
    plot_scenario_tornado, plot_dashboard, plot_regime_overlay,
)
from src.scenario_analysis import ScenarioEngine, SCENARIO_TEMPLATES
from src.cointegration import add_cointegration_features
from src.regime_detection import RegimeDetector
from src.feature_pruning import auto_prune_features

# Prophet is optional (heavy dependency)
try:
    from src.models_prophet import ProphetModel
    HAS_PROPHET = True
    print('✅ Prophet available')
except ImportError:
    HAS_PROPHET = False
    print('⚠️  Prophet not installed — ProphetModel will be skipped')

print('✅ All imports OK')

In [ ]:
# ── Configuration ─────────────────────────────────────────────────────────
CFG = {
    'start_date':        '2010-01-01',
    'forecast_horizon':  22,              # 1-month ahead (trading days)
    'all_horizons':      [1, 5, 22, 66],  # 1-day, 1-week, 1-month, 3-month
    'lags':              [1, 5, 22],      # look-back periods
    'initial_train_size': 504,            # ~2 years
    'cv_step_size':       22,
    'holdout_size':       252,            # ~1 year OOS backtest
    'ci_alpha':           0.80,           # confidence interval width
    'optuna_trials':      50,             # set to 0 to skip tuning
    'fred_api_key':       os.environ.get('FRED_API_KEY', None),
    'nasdaq_api_key':     os.environ.get('NASDAQ_DATA_LINK_API_KEY', None),
    'random_seed':        42,
    'output_dir':         './outputs',
}
os.makedirs(CFG['output_dir'], exist_ok=True)
print('Configuration:', CFG)

## 2. Data Ingestion

Data is downloaded from three sources:
- **yfinance**: Copper (HG=F), DXY, Gold, Aluminium, Oil, CNY/USD, S&P 500, Shanghai
- **FRED API**: Industrial production, real yields, inflation breakeven, M2
- **CFTC COT**: Commercial/non-commercial positioning, open interest, speculative ratio

If no API keys are available, synthetic placeholder series are used (safe for testing).

In [ ]:
df_raw = load_data(
    start=CFG['start_date'],
    fred_api_key=CFG['fred_api_key'],
    include_cot=True,
    nasdaq_api_key=CFG['nasdaq_api_key'],
)
print(f'Dataset shape: {df_raw.shape}')
print(f'Date range: {df_raw.index.min().date()} → {df_raw.index.max().date()}')
print(f'\nColumns: {list(df_raw.columns)}')
df_raw.tail()

In [ ]:
# Summary statistics
df_raw.describe().round(2)


In [ ]:
# Missing data audit
missing = df_raw.isna().sum()
missing_pct = (missing / len(df_raw) * 100).round(1)
audit = pd.DataFrame({'missing_count': missing, 'missing_pct': missing_pct})
print('Missing data audit:')
print(audit[audit['missing_count'] > 0].to_string() if (audit['missing_count'] > 0).any() else 'No missing data')


## 3. Exploratory Data Analysis

### 3.1 Copper price history


In [ ]:
fig_price = plot_price_history(df_raw)
fig_price.show()


### 3.2 Correlation with copper price


In [ ]:
# Pairwise correlations of all series with copper price
corr_all = df_raw.corr()['copper_price'].drop('copper_price').sort_values()
import plotly.express as px
fig_corr_bar = px.bar(
    x=corr_all.values, y=corr_all.index,
    orientation='h', title='Pearson Correlation with Copper Price',
    labels={'x': 'Pearson r', 'y': 'Series'},
    template='plotly_white',
    color=corr_all.values,
    color_continuous_scale='RdBu',
    color_continuous_midpoint=0,
)
fig_corr_bar.show()


### 3.3 Return distribution


In [ ]:
log_ret = np.log(df_raw['copper_price'] / df_raw['copper_price'].shift(1)).dropna()

import plotly.graph_objects as go
fig_dist = go.Figure()
fig_dist.add_trace(go.Histogram(
    x=log_ret, nbinsx=80, name='Daily log return',
    marker_color='#b87333', opacity=0.75,
))
fig_dist.update_layout(
    title='Distribution of Daily Copper Log Returns',
    xaxis_title='Log Return', yaxis_title='Count',
    template='plotly_white',
)
fig_dist.show()
print(f'Skewness: {log_ret.skew():.3f}  |  Kurtosis: {log_ret.kurt():.3f}')
print(f'Ann. volatility: {log_ret.std() * np.sqrt(252):.1%}')


## 4. Cointegration Analysis & Regime Detection

### 4.1 Cointegration tests

We test copper's long-run equilibrium relationships with gold, aluminium, oil, DXY, and CNY using the Engle-Granger two-step procedure. For cointegrated pairs, rolling error-correction terms (ECTs) are computed as features.

In [ ]:
# Run cointegration tests and add ECT features
df_aug, coint_results = add_cointegration_features(df_raw)

# Display results
coint_df = pd.DataFrame(coint_results).T
print('── Cointegration Test Results (Engle-Granger) ──')
print(coint_df.to_string())
print(f'\nCointegrated pairs: {[k for k, v in coint_results.items() if v["is_cointegrated"]]}')
print(f'ECT columns added: {[c for c in df_aug.columns if c.startswith("ect_")]}')

### 4.2 Regime detection (Hidden Markov Model)

A 3-state Gaussian HMM identifies latent market regimes from returns, volatility, and z-score. States are relabelled by ascending mean return: **0 = bear, 1 = sideways, 2 = bull**.

Regime labels are added as features (one-hot encoded) for the forecasting models.

In [ ]:
# Build preliminary features for regime detection (needs returns, vol, zscore)
feats_prelim = build_features(df_aug, lags=CFG['lags'], horizons=CFG['all_horizons'])

# Fit regime detector on development data (exclude holdout)
n_holdout = CFG['holdout_size']
regime_detector = RegimeDetector(n_regimes=3)

try:
    regime_detector.fit(feats_prelim.iloc[:-n_holdout])
    regime_labels = regime_detector.predict(feats_prelim)
    print(f'Regime distribution:\n{regime_labels.value_counts().sort_index()}')
    
    # Plot regime overlay on copper price
    fig_regime = plot_regime_overlay(df_aug['copper_price'], regime_labels)
    fig_regime.show()
except ImportError:
    print('⚠️  hmmlearn not installed — regime detection skipped')
    regime_labels = None
except Exception as e:
    print(f'⚠️  Regime detection failed: {e}')
    regime_labels = None

## 5. Feature Engineering

Features are built in five groups:

| Group | Features |
|---|---|
| **Price-derived** | Log returns (1/5/22d), realised vol, z-score vs 200-day MA, RSI-14, MACD, Bollinger width |
| **Cross-asset** | Gold/copper ratio, oil/copper ratio, Al–Cu spread %, DXY level & return, CNY/USD, S&P 500 |
| **Macro/calendar** | IP YoY, real yield, inflation breakeven, month sin/cos, CNY flag, quarter-end, US holidays, options expiry |
| **COT positioning** | Commercial net, non-commercial net, open interest, speculative ratio |
| **Structural** | Cointegration ECTs (rolling), HMM regime dummies |

Each base feature is lagged at 1, 5, and 22 days to preserve causality.

In [ ]:
feats = build_features(
    df_aug,
    lags=CFG['lags'],
    horizons=CFG['all_horizons'],
)

# Add regime features if available
if regime_labels is not None:
    feats['regime'] = regime_labels.reindex(feats.index)
    for i in range(3):
        feats[f'regime_{i}'] = (feats['regime'] == i).astype(float)

print(f'Feature matrix shape: {feats.shape}')
feat_cols = [c for c in feats.columns if not c.startswith('target_') and c != 'copper_price']
print(f'Number of features (pre-lag): {len([c for c in feat_cols if "_lag_" not in c])}')
print(f'Total features (incl. lags): {len(feat_cols)}')

In [ ]:
X, y_ret, y_price = split_features_targets(feats, horizon=CFG['forecast_horizon'])
print(f'X: {X.shape}  |  y: {y_ret.shape}')
print(f'Date range: {X.index.min().date()} → {X.index.max().date()}')


In [ ]:
# Feature–target correlation chart (top 25 features)
fig_feat_corr = plot_feature_correlations(X, y_ret, top_n=25)
fig_feat_corr.show()


## 6. Model Training & Hyper-parameter Tuning

### Models built
1. **Naive** — random-walk baseline (always predicts 0 return)
2. **Ridge** — linear benchmark with standard scaling
3. **XGBoost** — gradient-boosted trees (Optuna-tuned)
4. **LightGBM** — fast gradient boosting (Optuna-tuned)
5. **ARIMAX** — econometric model with macro exogenous features
6. **Prophet** — Bayesian structural time series (optional)
7. **Hybrid** — ARIMAX backbone + LightGBM residual correction
8. **Weighted Ensemble** — equal-weight average of XGBoost + LightGBM
9. **Stacking Ensemble** — meta-learner (Ridge) on walk-forward OOF predictions

Tuning uses `TimeSeriesSplit` cross-validation to prevent look-ahead.

In [ ]:
naive   = NaiveModel()
linear  = LinearModel()
xgb_mdl = XGBoostModel()
lgb_mdl = LGBMModel()
arimax  = ARIMAXModel()
hybrid  = HybridModel()

if HAS_PROPHET:
    prophet_mdl = ProphetModel()

# Split off the final holdout before any tuning
n_holdout = CFG['holdout_size']
X_dev, y_dev = X.iloc[:-n_holdout], y_ret.iloc[:-n_holdout]
X_hold, y_hold = X.iloc[-n_holdout:], y_ret.iloc[-n_holdout:]
price_hold = y_price.iloc[-n_holdout:]

print(f'Development set: {len(X_dev)} rows')
print(f'Holdout set:     {len(X_hold)} rows')

In [ ]:
# ── Optuna tuning (set optuna_trials=0 to skip) ──────────────────────────
import time

if CFG['optuna_trials'] > 0:
    print('Tuning XGBoost...')
    t0 = time.time()
    best_xgb = xgb_mdl.tune(X_dev, y_dev, n_trials=CFG['optuna_trials'])
    print(f'  Done in {time.time()-t0:.1f}s | best params: {best_xgb}')

    print('Tuning LightGBM...')
    t0 = time.time()
    best_lgb = lgb_mdl.tune(X_dev, y_dev, n_trials=CFG['optuna_trials'])
    print(f'  Done in {time.time()-t0:.1f}s | best params: {best_lgb}')
else:
    print('Skipping Optuna tuning (optuna_trials=0). Using default params.')


In [ ]:
# Weighted ensemble of the two gradient-boosting models
w_ensemble = EnsembleModel([xgb_mdl, lgb_mdl])

# Fit all base models on the development set
base_models = [naive, linear, xgb_mdl, lgb_mdl, arimax, hybrid]
if HAS_PROPHET:
    base_models.append(prophet_mdl)
base_models.append(w_ensemble)

for m in base_models:
    m.fit(X_dev, y_dev)
    print(f'Fitted: {m.name}')

# Stacking ensemble (uses walk-forward OOF internally)
print('\nFitting stacking ensemble (this may take a few minutes)...')
stack_base = [XGBoostModel(), LGBMModel(), ARIMAXModel()]
if HAS_PROPHET:
    stack_base.append(ProphetModel())

stacking = StackingEnsemble(
    base_models=stack_base,
    oof_initial_size=CFG['initial_train_size'],
    oof_step=CFG['cv_step_size'],
)
stacking.fit(X_dev, y_dev)
print(f'Fitted: {stacking.name}')

## 7. Feature Pruning (SHAP-based)

Automatically remove features in the bottom 20% by mean |SHAP| importance. Retrain and compare performance before/after pruning.

In [ ]:
# Auto-prune low-importance features using SHAP
try:
    prune_model = XGBoostModel()
    X_pruned, dropped, importance = auto_prune_features(
        prune_model, X_dev, y_dev, threshold='bottom_20pct'
    )
    print(f'Features before pruning: {X_dev.shape[1]}')
    print(f'Features after pruning:  {X_pruned.shape[1]}')
    print(f'Dropped {len(dropped)} features')
    print(f'\nTop 15 features by importance:')
    print(importance.head(15).to_string(index=False))

    # Update X, X_dev, X_hold with pruned features
    kept_cols = X_pruned.columns.tolist()
    X_dev_pruned = X_dev[kept_cols]
    X_hold_pruned = X_hold[kept_cols]
    X_pruned_full = X[kept_cols]
    USE_PRUNED = True
except Exception as e:
    print(f'Feature pruning skipped: {e}')
    X_dev_pruned = X_dev
    X_hold_pruned = X_hold
    X_pruned_full = X
    USE_PRUNED = False
    kept_cols = X_dev.columns.tolist()

## 8. Walk-Forward Cross-Validation

Expanding-window validation simulates live deployment:
- **Initial window**: 504 trading days (~2 years)
- **Step size**: 22 days (model re-fitted monthly)
- All predictions are strictly out-of-sample for their training window
- Metrics include **signal Sharpe** for trading signal quality

In [ ]:
# Re-instantiate all models fresh for CV (using pruned features)
cv_models = [
    NaiveModel(), LinearModel(), XGBoostModel(), LGBMModel(),
    ARIMAXModel(), HybridModel(),
    EnsembleModel([XGBoostModel(), LGBMModel()]),
]
if HAS_PROPHET:
    cv_models.append(ProphetModel())

print('Running walk-forward CV...')
summary, cv_results = compare_models(
    cv_models, X_dev_pruned, y_dev,
    initial_train_size=CFG['initial_train_size'],
    step_size=CFG['cv_step_size'],
    horizon=CFG['forecast_horizon'],
)
print('\n── Walk-Forward CV Metrics ──')
print(summary.to_string())

In [ ]:
# Visualise model comparison
fig_cmp = plot_model_comparison(summary, metric='rmse')
fig_cmp.show()


In [ ]:
fig_cmp_da = plot_model_comparison(summary, metric='directional_accuracy',
                                    title='Model Comparison — Directional Accuracy')
fig_cmp_da.show()


In [ ]:
# Best model CV predictions vs actuals
best_name = summary['rmse'].idxmin()
print(f'Best model by CV RMSE: {best_name}')
fig_cv = plot_cv_results(cv_results[best_name], model_name=best_name)
fig_cv.show()


## 9. Out-of-Sample Backtesting

Train on all development data; evaluate on the last ~252 trading days (one year).
This mimics a realistic forward-test scenario.

In [ ]:
oos_results = {}
oos_metrics = {}

# Re-instantiate and fit all models on pruned dev data for OOS test
oos_model_list = [
    NaiveModel(), LinearModel(), XGBoostModel(), LGBMModel(),
    ARIMAXModel(), HybridModel(),
    EnsembleModel([XGBoostModel(), LGBMModel()]),
]
if HAS_PROPHET:
    oos_model_list.append(ProphetModel())

for m in oos_model_list:
    m.fit(X_dev_pruned, y_dev)
    oos_df, met = out_of_sample_backtest(
        m, X_pruned_full, y_ret,
        holdout_size=n_holdout,
        horizon=CFG['forecast_horizon'],
    )
    oos_results[m.name] = oos_df
    oos_metrics[m.name] = met

oos_summary = pd.DataFrame(oos_metrics).T
print('── Out-of-Sample Backtest Metrics ──')
print(oos_summary.to_string())

In [ ]:
# OOS actual vs predicted for best model
best_oos = oos_summary['rmse'].idxmin()
fig_oos = plot_cv_results(oos_results[best_oos], model_name=f'{best_oos} (OOS)')
fig_oos.show()


## 10. Forecast with 80% Confidence Interval

Multiple models provide prediction intervals:
- **LightGBM quantile regression** (10th / 50th / 90th percentiles)
- **ARIMAX** built-in confidence bands
- **Prophet** built-in uncertainty (if available)

The CI is converted back to price space: `price = last_price × exp(forecast_return)`

In [ ]:
# Fit quantile forecaster on pruned development set
q_model = QuantileForecaster(alpha=CFG['ci_alpha'])
q_model.fit(X_dev_pruned, y_dev)

# Forecast the holdout period (known actuals for validation)
q_preds_ret = q_model.predict(X_hold_pruned)
last_price = df_aug['copper_price'].iloc[-(n_holdout+1)]

forecast_df = pd.DataFrame({
    'date': X_hold.index,
    'lower':  last_price * np.exp(q_preds_ret['lower'].values),
    'median': last_price * np.exp(q_preds_ret['median'].values),
    'upper':  last_price * np.exp(q_preds_ret['upper'].values),
})
forecast_df = forecast_df.set_index('date')
print(f'Forecast period: {forecast_df.index.min().date()} → {forecast_df.index.max().date()}')

# Also get ARIMAX prediction intervals
arimax_ci_model = ARIMAXModel()
arimax_ci_model.fit(X_dev_pruned, y_dev)
arimax_ci = arimax_ci_model.predict_interval(X_hold_pruned, alpha=CFG['ci_alpha'])
print(f'\nARIMAX prediction intervals available: {arimax_ci.columns.tolist()}')

forecast_df.head()

In [ ]:
fig_fc = plot_forecast_with_ci(df_raw['copper_price'], forecast_df.reset_index())
fig_fc.show()


In [ ]:
# Coverage check: what fraction of actuals fall within the 80% CI?
actual_hold = df_aug['copper_price'].reindex(X_hold.index)
inside = ((actual_hold >= forecast_df['lower']) & (actual_hold <= forecast_df['upper'])).mean()
print(f'80% CI actual coverage (LightGBM quantile): {inside:.1%}  (target: 80%)')

## 11. SHAP Explainability

SHAP (SHapley Additive exPlanations) values quantify each feature's contribution to the
model's prediction.  We use `shap.TreeExplainer` which is exact and fast for tree ensembles.

In [ ]:
try:
    import shap
    shap.initjs()

    # Refit XGBoost on pruned features for SHAP
    xgb_shap = XGBoostModel()
    xgb_shap.fit(X_dev_pruned, y_dev)
    explainer = shap.TreeExplainer(xgb_shap._model)
    shap_values = explainer.shap_values(X_hold_pruned)

    fig_shap = plot_shap_summary(shap_values, list(X_hold_pruned.columns), top_n=20)
    fig_shap.show()

    # SHAP beeswarm (requires matplotlib)
    try:
        import matplotlib.pyplot as plt
        shap.summary_plot(shap_values, X_hold_pruned, max_display=20, show=False)
        plt.tight_layout()
        plt.savefig(os.path.join(CFG['output_dir'], 'shap_beeswarm.png'), dpi=150, bbox_inches='tight')
        plt.show()
        print('SHAP beeswarm saved.')
    except Exception as e:
        print(f'Beeswarm plot skipped: {e}')
except Exception as e:
    print(f'SHAP analysis skipped: {e}')

In [ ]:
# Top 10 features by mean |SHAP|
try:
    mean_abs_shap = pd.Series(
        np.abs(shap_values).mean(axis=0),
        index=X_hold_pruned.columns
    ).sort_values(ascending=False)
    print('Top 10 features by mean |SHAP|:')
    print(mean_abs_shap.head(10).to_string())
except NameError:
    print('SHAP values not available.')

## 12. Interactive Plotly Visualisations

In [ ]:
# Full dashboard: price history + best-model CV results
best_cv_df = cv_results[best_name]
fig_dashboard = plot_dashboard(df_aug, best_cv_df, model_name=best_name)
fig_dashboard.show()

In [ ]:
# Rolling Sharpe-equivalent for the best OOS model signal
best_oos_name = oos_summary['rmse'].idxmin()
oos_best = oos_results[best_oos_name]
signal_ret_oos = np.sign(oos_best['y_pred']) * oos_best['y_true']
rolling_sharpe = (signal_ret_oos.rolling(60).mean() /
                  signal_ret_oos.rolling(60).std().replace(0, np.nan)) * np.sqrt(252 / CFG['forecast_horizon'])

import plotly.graph_objects as go
fig_sharpe = go.Figure(go.Scatter(
    x=rolling_sharpe.index, y=rolling_sharpe,
    name='Rolling Sharpe (60-day)',
    line=dict(color='#b87333'),
))
fig_sharpe.add_hline(y=0, line_width=1, line_dash='dash', line_color='grey')
fig_sharpe.update_layout(
    title=f'Rolling 60-Day Signal Sharpe — {best_oos_name}',
    template='plotly_white',
    yaxis_title='Annualised Sharpe ratio',
)
fig_sharpe.show()

## 13. Scenario Analysis

The `ScenarioEngine` takes the trained ensemble model and the most recent feature vector,
then applies additive shocks to simulate what-if conditions.

### Built-in scenarios
| Scenario | Description |
|---|---|
| `bull_strong` | DXY −5%, real yields −50 bps, IP +3 pp |
| `bear_strong` | DXY +5%, real yields +50 bps, IP −3 pp |
| `china_demand_surge` | IP +5 pp, CNY appreciation |
| `supply_disruption` | Higher vol, short-term squeeze |
| `comex_inventory_drop_40pct` | Price above 200-day average, higher vol |
| `high_inflation` | Inflation breakeven +100 bps, lower real yields |
| `us_tariff_shock` | DXY +3%, equity drawdown, higher vol |

In [ ]:
# Use the latest feature vector (pruned)
latest_features = X_pruned_full.tail(1)
current_copper = float(df_aug['copper_price'].iloc[-1])

# Use the best-performing ensemble for scenarios
scenario_model = EnsembleModel([XGBoostModel(), LGBMModel()])
scenario_model.fit(X_dev_pruned, y_dev)

engine = ScenarioEngine(
    model=scenario_model,
    feature_template=latest_features,
    copper_price_current=current_copper,
    horizon=CFG['forecast_horizon'],
)
print(f'Current copper price: ${current_copper:,.0f}/t')
print(f'Baseline {CFG["forecast_horizon"]}-day forecast: ${engine.base_price:,.0f}/t')

In [ ]:
# Run all pre-defined scenarios
scenario_report = engine.report()
print('\n── Scenario Report ──')
print(scenario_report.to_string())


In [ ]:
# Tornado chart
fig_tornado = plot_scenario_tornado(
    base_forecast=engine.base_price,
    scenario_results={row.Index: row.scenario_price for row in scenario_report.itertuples()},
)
fig_tornado.show()


In [ ]:
# Sensitivity sweep: DXY shock
dxy_shocks = np.linspace(-0.10, 0.10, 21)
sweep_dxy = engine.sweep('dxy_ret_22d', dxy_shocks.tolist(), label='DXY 22d return shock')

fig_sweep = go.Figure(go.Scatter(
    x=sweep_dxy['shock'], y=sweep_dxy['forecast_price'],
    mode='lines+markers', line=dict(color='#b87333'),
    name='Forecast price',
))
fig_sweep.add_hline(y=engine.base_price, line_dash='dash', line_color='grey',
                    annotation_text='Baseline')
fig_sweep.update_layout(
    title='Copper Price Sensitivity to DXY Return Shock',
    xaxis_title='DXY 22d return shock (additive)',
    yaxis_title='Forecast Copper Price ($/t)',
    template='plotly_white',
)
fig_sweep.show()


In [ ]:
# Custom scenario: combined geopolitical / tariff shock
custom_scenario = {
    'dxy_ret_22d':          0.04,
    'sp500_ret_22d':        -0.08,
    'copper_vol_22d':       0.05,
    'real_yield_change_22d': 0.3,
}
result = engine.run('geo_tariff_shock', shocks=custom_scenario)
print('Custom scenario result:')
for k, v in result.items():
    print(f'  {k}: {v}')


## 14. Export Results

All outputs are saved to `./outputs/` as CSV and JSON for downstream use
(e.g. Power BI, Tableau, or automated reporting pipelines).

In [ ]:
import json
from datetime import date

out_dir = CFG['output_dir']

# 1. Forecast with CI
forecast_df.reset_index().rename(columns={'index': 'date'}).to_csv(
    os.path.join(out_dir, 'forecast_ci.csv'), index=False
)

# 2. OOS backtest results
for name, df_oos in oos_results.items():
    safe_name = name.replace(' ', '_').replace('(', '').replace(')', '')
    df_oos.to_csv(os.path.join(out_dir, f'oos_{safe_name}.csv'))

# 3. Model comparison summary
oos_summary.to_csv(os.path.join(out_dir, 'model_comparison.csv'))

# 4. Scenario report
scenario_report.to_csv(os.path.join(out_dir, 'scenario_report.csv'))

# 5. Cointegration results
coint_df.to_csv(os.path.join(out_dir, 'cointegration_results.csv'))

# 6. JSON summary (for APIs / dashboards)
summary_json = {
    'generated_at':      date.today().isoformat(),
    'current_price':     round(current_copper, 2),
    'baseline_forecast': round(engine.base_price, 2),
    'horizon_days':      CFG['forecast_horizon'],
    'best_model':        best_oos_name,
    'oos_metrics':       oos_metrics[best_oos_name],
    'scenarios':         scenario_report.reset_index().to_dict(orient='records'),
}
with open(os.path.join(out_dir, 'forecast_summary.json'), 'w') as f:
    json.dump(summary_json, f, indent=2)

print(f'All outputs saved to {out_dir}/')
print('Files:', os.listdir(out_dir))

---
## Summary

| Step | Completed |
|---|---|
| Data ingestion (yfinance + FRED + COT) | ✅ |
| Cointegration analysis (Engle-Granger + ECT features) | ✅ |
| Regime detection (3-state HMM) | ✅ |
| Feature engineering (price, cross-asset, macro, calendar, COT, structural) | ✅ |
| Model training: Naive, Ridge, XGBoost, LightGBM, ARIMAX, Prophet, Hybrid, Ensembles | ✅ |
| Feature pruning (SHAP-based bottom 20% removal) | ✅ |
| Optuna hyper-parameter tuning | ✅ |
| Walk-forward cross-validation (with signal Sharpe) | ✅ |
| Out-of-sample backtest (last 12 months) | ✅ |
| Forecast with 80% CI (quantile + ARIMAX intervals) | ✅ |
| SHAP explainability | ✅ |
| Interactive Plotly visualisations | ✅ |
| Scenario analysis | ✅ |
| JSON / CSV export | ✅ |